In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.metrics import mean_squared_error,r2_score,accuracy_score,confusion_matrix,classification_report
from sklearn.preprocessing import MinMaxScaler,StandardScaler,LabelEncoder
from sklearn.ensemble import RandomForestRegressor,RandomForestClassifier
from xgboost import XGBRegressor,XGBClassifier

In [ ]:
df = pd.read_csv("/content/Delivery_Logistics.csv")

In [ ]:
df.sample(5)

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost
23409,23410.0,blue dart,electronics,bike,two day,west,rainy,80.8,19.59,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016,no,delivered,5,462.77
11711,11712.0,amazon logistics,electronics,ev bike,express,west,rainy,220.3,20.95,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000006,yes,delayed,2,1214.35
4281,4282.0,ecom express,pharmacy,ev bike,two day,north,foggy,215.6,12.09,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016,no,delivered,4,1114.27
12394,12395.0,shadowfax,documents,ev bike,same day,west,clear,226.1,14.15,1970-01-01 00:00:00.000000007,1970-01-01 00:00:00.000000008,no,delivered,4,1272.95
21296,21297.0,ekart,pharmacy,truck,same day,west,hot,84.0,8.27,1970-01-01 00:00:00.000000004,1970-01-01 00:00:00.000000008,no,delivered,5,544.81


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   delivery_id          25000 non-null  float64
 1   delivery_partner     25000 non-null  object 
 2   package_type         25000 non-null  object 
 3   vehicle_type         25000 non-null  object 
 4   delivery_mode        25000 non-null  object 
 5   region               25000 non-null  object 
 6   weather_condition    25000 non-null  object 
 7   distance_km          25000 non-null  float64
 8   package_weight_kg    25000 non-null  float64
 9   delivery_time_hours  25000 non-null  object 
 10  expected_time_hours  25000 non-null  object 
 11  delayed              25000 non-null  object 
 12  delivery_status      25000 non-null  object 
 13  delivery_rating      25000 non-null  int64  
 14  delivery_cost        25000 non-null  float64
dtypes: float64(4), int64(1), object(10)


In [ ]:
df['delivery_time_hours'] = pd.to_datetime(df['delivery_time_hours']).astype('int64')/1e9/3600

In [ ]:
df['expected_time_hours'] = pd.to_datetime(df['expected_time_hours']).astype('int64')/1e9/3600

In [ ]:
df.drop(columns=['delivery_id'],inplace=True)

In [ ]:
le = LabelEncoder()
df['delivery_partner'] = le.fit_transform(df['delivery_partner'])
df['package_type'] = le.fit_transform(df['package_type'])
df['delivery_mode'] = le.fit_transform(df['delivery_mode'])
df['delivery_status'] = le.fit_transform(df['delivery_status'])

In [ ]:
df = pd.get_dummies(df, columns=[
    'vehicle_type',
    'region',
    'weather_condition'
])

In [ ]:
df['delayed'] = df['delayed'].map({'yes': 1, 'no': 0})

In [ ]:
df['time_diff'] = df['delivery_time_hours'] - df['expected_time_hours']
df['speed'] = df['distance_km'] / df['delivery_time_hours']
df['cost_per_km'] = df['delivery_cost'] / df['distance_km']

In [ ]:
df.sample(5)

,delivery_partner,package_type,delivery_mode,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost,vehicle_type_bike,vehicle_type_ev bike,vehicle_type_ev van,vehicle_type_scooter,vehicle_type_truck,vehicle_type_van,region_central,region_east,region_north,region_south,region_west,weather_condition_clear,weather_condition_cold,weather_condition_foggy,weather_condition_hot,weather_condition_rainy,weather_condition_stormy,time_diff,speed,cost_per_km
12107,5,2,0,224.5,3.73,1.666667e-12,1.666667e-12,0,1,5,1183.69,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,0.000000e+00,1.347000e+14,5.272561
4093,6,6,3,265.5,31.55,2.500000e-12,4.444444e-12,0,1,3,1422.15,False,False,False,False,False,True,True,False,False,False,False,False,True,False,False,False,False,-1.944444e-12,1.062000e+14,5.356497
20028,3,2,0,220.6,47.63,2.500000e-12,1.666667e-12,1,2,1,1295.89,False,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,8.333333e-13,8.824000e+13,5.874388
3698,5,4,1,188.4,22.99,2.222222e-12,2.222222e-12,1,2,1,1110.97,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,0.000000e+00,8.478000e+13,5.896868
234,6,5,2,267.0,26.49,3.055556e-12,6.666667e-12,0,1,3,1414.47,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,-3.611111e-12,8.738182e+13,5.297640


In [ ]:
df.to_csv('preprocess data.csv',index=False)

In [ ]:
df = pd.read_csv("/content/preprocess data.csv")

In [ ]:
x = df.drop(columns=['delayed'])
y = df['delayed']

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   delivery_partner          25000 non-null  int64  
 1   package_type              25000 non-null  int64  
 2   delivery_mode             25000 non-null  int64  
 3   distance_km               25000 non-null  float64
 4   package_weight_kg         25000 non-null  float64
 5   delivery_time_hours       25000 non-null  float64
 6   expected_time_hours       25000 non-null  float64
 7   delayed                   25000 non-null  int64  
 8   delivery_status           25000 non-null  int64  
 9   delivery_rating           25000 non-null  int64  
 10  delivery_cost             25000 non-null  float64
 11  vehicle_type_bike         25000 non-null  bool   
 12  vehicle_type_ev bike      25000 non-null  bool   
 13  vehicle_type_ev van       25000 non-null  bool   
 14  vehicl

In [ ]:
df.isnull().sum()

,0
delivery_partner,0
package_type,0
delivery_mode,0
distance_km,0
package_weight_kg,0
delivery_time_hours,0
expected_time_hours,0
delayed,0
delivery_status,0
delivery_rating,0


In [ ]:
import numpy as np
x_train.replace([np.inf, -np.inf], np.nan, inplace=True)
x_test.replace([np.inf, -np.inf], np.nan, inplace=True)

train_nan_indices = x_train.isnull().any(axis=1)
x_train = x_train[~train_nan_indices]
y_train = y_train[~train_nan_indices]

test_nan_indices = x_test.isnull().any(axis=1)
x_test = x_test[~test_nan_indices]
y_test = y_test[~test_nan_indices]

model.fit(x_train,y_train)

RandomForestClassifier(max_depth=10, random_state=42)

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
print("Accuracy : ", accuracy_score(y_test,y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy :  1.0

Confusion Matrix:
 [[3629    0]
 [   0 1323]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      3629
           1       1.00      1.00      1.00      1323

    accuracy                           1.00      4952
   macro avg       1.00      1.00      1.00      4952
weighted avg       1.00      1.00      1.00      4952



### finally using pipeline for training which automatically handle categorical and numerical features

In [ ]:
df = pd.read_csv('/content/Delivery_Logistics.csv')
df['expected_time_hours'] = pd.to_datetime(df['expected_time_hours']).astype('int64') / 1e9 / 3600
df['delayed'] = df['delayed'].map({'yes': 1, 'no': 0})
df.drop(columns=[
    'delivery_id',
    'delivery_status'
], inplace=True)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

categorical_cols = [
    'delivery_partner',
    'package_type',
    'delivery_mode',
    'vehicle_type',
    'region',
    'weather_condition'
]

numerical_cols = [
    'distance_km',
    'package_weight_kg',
    'expected_time_hours',
    'delivery_cost'
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   delivery_partner     25000 non-null  object 
 1   package_type         25000 non-null  object 
 2   vehicle_type         25000 non-null  object 
 3   delivery_mode        25000 non-null  object 
 4   region               25000 non-null  object 
 5   weather_condition    25000 non-null  object 
 6   distance_km          25000 non-null  float64
 7   package_weight_kg    25000 non-null  float64
 8   delivery_time_hours  25000 non-null  object 
 9   expected_time_hours  25000 non-null  float64
 10  delayed              25000 non-null  int64  
 11  delivery_rating      25000 non-null  int64  
 12  delivery_cost        25000 non-null  float64
dtypes: float64(4), int64(2), object(7)
memory usage: 2.5+ MB


In [ ]:
x = df[categorical_cols + numerical_cols]
y = df['delayed']

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

In [ ]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['delivery_partner',
                                                   'package_type',
                                                   'delivery_mode',
                                                   'vehicle_type', 'region',
                                                   'weather_condition']),
                                                 ('num', 'passthrough',
                                                  ['distance_km',
                                                   'package_weight_kg',
                                                   'expected_time_hours',
                                                   'delivery_cost'])])),
                ('model', RandomForestClassifier(random_state=42))])

In [ ]:
y_pred = pipeline.predict(x_test)

In [ ]:
print('accuracy : ' , accuracy_score(y_test,y_pred))

accuracy :  0.8924


In [ ]:
import joblib
joblib.dump(pipeline, 'pipeline_model.pkl')

['pipeline_model.pkl']

In [ ]:
pipeline1 = joblib.load('pipeline_model.pkl')

In [ ]:
sample_input = {
    "delivery_partner": "fedex",
    "package_type": "clothing",
    "delivery_mode": "express",
    "vehicle_type": "truck",
    "region": "north",
    "weather_condition": "rainy",
    "distance_km": 250,
    "package_weight_kg": 10,
    "expected_time_hours": 6,
    "delivery_cost": 1200
}

In [ ]:
input_df = pd.DataFrame([sample_input])

In [ ]:
prediction = pipeline1.predict(input_df)

In [ ]:
if prediction[0] == 1:
  print("delayed delivery")
else:
  print("on-time devery")

on-time devery
